In [0]:
data = [

    ("Cardiology", "2024-01", 50000),

    ("Cardiology", "2024-02", 55000),

    ("Cardiology", "2024-03", 53000),
 
    ("Neurology", "2024-01", 40000),

    ("Neurology", "2024-02", 45000),

    ("Neurology", "2024-03", 47000),
 
    ("Orthopedics", "2024-01", 30000),

    ("Orthopedics", "2024-02", 32000),

    ("Orthopedics", "2024-03", 31000),
 
    ("Dermatology", "2024-01", 20000),

    ("Dermatology", "2024-02", 25000),

    ("Dermatology", "2024-03", 28000)

]
 
columns = ["department", "month", "revenue"]
 
df = spark.createDataFrame(data, columns)
 

In [0]:
df.show()

+-----------+-------+-------+
| department|  month|revenue|
+-----------+-------+-------+
| Cardiology|2024-01|  50000|
| Cardiology|2024-02|  55000|
| Cardiology|2024-03|  53000|
|  Neurology|2024-01|  40000|
|  Neurology|2024-02|  45000|
|  Neurology|2024-03|  47000|
|Orthopedics|2024-01|  30000|
|Orthopedics|2024-02|  32000|
|Orthopedics|2024-03|  31000|
|Dermatology|2024-01|  20000|
|Dermatology|2024-02|  25000|
|Dermatology|2024-03|  28000|
+-----------+-------+-------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df.withColumn("runningtotal", sum("revenue").over(Window.partitionBy("department").orderBy("month"))).show()

+-----------+-------+-------+------------+
| department|  month|revenue|runningtotal|
+-----------+-------+-------+------------+
| Cardiology|2024-01|  50000|       50000|
| Cardiology|2024-02|  55000|      105000|
| Cardiology|2024-03|  53000|      158000|
|Dermatology|2024-01|  20000|       20000|
|Dermatology|2024-02|  25000|       45000|
|Dermatology|2024-03|  28000|       73000|
|  Neurology|2024-01|  40000|       40000|
|  Neurology|2024-02|  45000|       85000|
|  Neurology|2024-03|  47000|      132000|
|Orthopedics|2024-01|  30000|       30000|
|Orthopedics|2024-02|  32000|       62000|
|Orthopedics|2024-03|  31000|       93000|
+-----------+-------+-------+------------+



In [0]:
df.withColumn("runningtotal", sum("revenue").over(Window.partitionBy("month").orderBy("month"))).show()

+-----------+-------+-------+------------+
| department|  month|revenue|runningtotal|
+-----------+-------+-------+------------+
| Cardiology|2024-01|  50000|      140000|
|  Neurology|2024-01|  40000|      140000|
|Orthopedics|2024-01|  30000|      140000|
|Dermatology|2024-01|  20000|      140000|
| Cardiology|2024-02|  55000|      157000|
|  Neurology|2024-02|  45000|      157000|
|Orthopedics|2024-02|  32000|      157000|
|Dermatology|2024-02|  25000|      157000|
| Cardiology|2024-03|  53000|      159000|
|  Neurology|2024-03|  47000|      159000|
|Orthopedics|2024-03|  31000|      159000|
|Dermatology|2024-03|  28000|      159000|
+-----------+-------+-------+------------+



In [0]:
df_lag = df.withColumn("previousrevenue", lag("revenue").over(Window.partitionBy("department").orderBy("month")))
# df_lag.show()
df_change = df_lag.withColumn("change", col("revenue") - col("previousrevenue"))
# df_change.show()
df_percent = df_change.withColumn("percentage", (col("revenue") - col("previousrevenue")) / col("previousrevenue") * 100)
df_percent.show()

+-----------+-------+-------+---------------+------+-------------------+
| department|  month|revenue|previousrevenue|change|         percentage|
+-----------+-------+-------+---------------+------+-------------------+
| Cardiology|2024-01|  50000|           NULL|  NULL|               NULL|
| Cardiology|2024-02|  55000|          50000|  5000|               10.0|
| Cardiology|2024-03|  53000|          55000| -2000|-3.6363636363636362|
|Dermatology|2024-01|  20000|           NULL|  NULL|               NULL|
|Dermatology|2024-02|  25000|          20000|  5000|               25.0|
|Dermatology|2024-03|  28000|          25000|  3000|               12.0|
|  Neurology|2024-01|  40000|           NULL|  NULL|               NULL|
|  Neurology|2024-02|  45000|          40000|  5000|               12.5|
|  Neurology|2024-03|  47000|          45000|  2000|  4.444444444444445|
|Orthopedics|2024-01|  30000|           NULL|  NULL|               NULL|
|Orthopedics|2024-02|  32000|          30000|  2000

In [0]:
df.withColumn("rank", dense_rank().over(Window.orderBy(desc("revenue")))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+-------+----+
| department|  month|revenue|rank|
+-----------+-------+-------+----+
| Cardiology|2024-02|  55000|   1|
| Cardiology|2024-03|  53000|   2|
| Cardiology|2024-01|  50000|   3|
|  Neurology|2024-03|  47000|   4|
|  Neurology|2024-02|  45000|   5|
|  Neurology|2024-01|  40000|   6|
|Orthopedics|2024-02|  32000|   7|
|Orthopedics|2024-03|  31000|   8|
|Orthopedics|2024-01|  30000|   9|
|Dermatology|2024-03|  28000|  10|
|Dermatology|2024-02|  25000|  11|
|Dermatology|2024-01|  20000|  12|
+-----------+-------+-------+----+

